# Kaggle (T4 x2, 2 hours): Gemma 3 27B on ten failed questions, with each of the 30 Sept fixes

`gemma3:27b-it-qat` answers ten questions the 29 Sept 27B run got wrong (one per root cause) with each fix
switched on in turn, through `scripts/legal_data/ab_fixes.py`. **No judge**: download the results zip from Output
and send it back for grading. The reference is the judged 29 Sept 27B score, which had none of these fixes.

| Arm | `--variant` | Aimed at |
|---|---|---|
| all | every fix together | how far the fixes go together |
| extract | `completeness` | a condition or exception left out although the section was retrieved (IL-063, IL-093, IL-301, IL-030) |
| toc_xref | `toc,xref` | right law, wrong section (IL-033, IL-034, IL-259) |
| whole_sections | `whole_sections` | the part of a split section that holds the exception |
| doctrines | `doctrines` | case-law doctrine and amendment timelines (IL-180, IL-344, IL-058) |

**Time.** The 29 Sept run took 40 s per question on this hardware; the fixes add one to three calls per question,
so about 55 minutes for 5 arms x 10 questions, after about 20 minutes of setup (packages, the 17 GB model, the
corpus). A time guard stops answering `STOP_ANSWERING_AFTER_H` after the start; the arms run in the order above, so
if time runs short the last one is cut. The comparison and the zip are written either way.

**Before running:** Settings -> Accelerator: **GPU T4 x2**; Internet on; Add-ons -> Secrets: `GITHUB_TOKEN`
(optionally `HF_TOKEN`); Input: the corpus dataset (`legal_corpus_vectordb`), as in the earlier runs.
Then Save Version -> Save & Run All, and download `ab_fixes_27b_results.zip` from Output.


In [ ]:
import time
NOTEBOOK_STARTED = time.time()   # the time guard counts from here

BRANCH = "main"
MODEL = {"tag": "gemma3:27b-it-qat", "thinking": False}
ARMS = "all,extract,toc_xref,whole_sections,doctrines"  # most informative first; add ",baseline" for a same-day reference (~7 min)
STOP_ANSWERING_AFTER_H = 1.75   # stop answering this long after the start (2 h session - report and zip - margin)
CONTEXT_LENGTH = 16384  # tokens per call (Ollama num_ctx): prompt + answer
ANSWER_MAX_TOKENS = 6144
REQUEST_TIMEOUT_S = 900  # per model call
RETRIEVAL_DEVICE = "cuda"  # embedder + reranker; falls back to the CPU by itself if the GPUs are full
RUN_NAME = "ab_fixes_27b"


## 1. Code, packages and the unit tests

In [ ]:
import os, pathlib, shutil, subprocess, json, time, urllib.request
from kaggle_secrets import UserSecretsClient

def sh(command, log=None, check=True, env=None):
    """Runs a shell command with its output streamed live (and appended to `log`); raises on
    failure when check=True -- unlike `!command`, which carries on silently."""
    proc = subprocess.Popen(["bash", "-o", "pipefail", "-c", command], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    out = open(log, "a", encoding="utf-8") if log else None
    for line in proc.stdout:
        print(line, end="", flush=True)
        if out:
            out.write(line)
    if out:
        out.close()
    if proc.wait() and check:
        raise RuntimeError(f"exit code {proc.returncode}: {command}")
    return proc.returncode

assert shutil.which("nvidia-smi"), (
    "NO GPU on this machine (nvidia-smi not found). Settings -> Accelerator: GPU T4 x2 -- for a saved version, "
    "set it in the editor before Save Version -> Save & Run All.")
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")
gpus = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()
assert len(gpus) >= 2, "Settings -> Accelerator: GPU T4 x2 -- the 27B model needs both GPUs"

token = UserSecretsClient().get_secret("GITHUB_TOKEN")
# Hugging Face token (Add-ons -> Secrets -> HF_TOKEN, read access): the embedder and reranker download
# with it -- higher rate limits, no "unauthenticated requests" warning. Optional: without it they still download.
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HUGGING_FACE_HUB_TOKEN"] = os.environ["HF_TOKEN"]
    print("HF_TOKEN: set")
except Exception:
    print("HF_TOKEN: not set (optional) -- Hugging Face downloads run unauthenticated")
REPO = "/kaggle/working/AI-IZ"
if not pathlib.Path(REPO, ".git").exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", BRANCH,
                    f"https://{token}@github.com/zananiri/AI-IZ.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "remote", "set-url", "origin", "https://github.com/zananiri/AI-IZ.git"], check=True)
os.chdir(REPO)
RUN_DIR = pathlib.Path(f"data/legal/eval/{RUN_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
sh("git log --oneline -1")

sh('pip install -q uv')
sh(f'uv pip install -q --system -e ".[legal,legal-data]" pytest')

# The code this run depends on, tested first (a failure is printed, not fatal; the full suite has a few
# known failures unrelated to the Legal eval).
sh("python -m pytest -q -p no:cacheprovider tests/unit/test_legal_corpus_lexical.py "
   "tests/unit/test_legal_corpus_retrieval.py tests/unit/test_legal_eval_repairs.py tests/unit/test_llm_client.py "
   "tests/unit/test_legal_corpus_navigation.py",
   log=RUN_DIR / "unit_tests.log", check=False)

## 2. Ollama and the model

`OLLAMA_MAX_LOADED_MODELS=1`: one model at a time. Flash attention + an 8-bit KV cache halve the 16k context's
memory. `OLLAMA_KEEP_ALIVE=-1`: the model stays loaded between the scripts, so it isn't reloaded after the
retrieval models have taken their share of a GPU. The 27B (~18 GB) doesn't fit one T4 (15 GB), so Ollama
splits it over both.

In [ ]:
sh("apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils lshw > /dev/null")
if not shutil.which("ollama"):
    sh("curl -fsSL https://ollama.com/install.sh | sh > /dev/null")

OLLAMA_LOG = RUN_DIR / "ollama.log"
if subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
    ollama = subprocess.Popen(["ollama", "serve"], stdout=open(OLLAMA_LOG, "a"), stderr=subprocess.STDOUT,
                              env={**os.environ, "OLLAMA_MAX_LOADED_MODELS": "1", "OLLAMA_FLASH_ATTENTION": "1",
                                   "OLLAMA_KV_CACHE_TYPE": "q8_0", "OLLAMA_KEEP_ALIVE": "-1"})
    for _ in range(60):
        if not subprocess.run(["curl", "-sf", "http://localhost:11434/api/tags"], capture_output=True).returncode:
            break
        time.sleep(2)

sh(f"ollama pull {MODEL['tag']}")
sh("ollama list")

## 3. The bulk legal corpus (attached input) and its BM25 index

In [ ]:
# The corpus database: category folders (laws/, procedural_rules/) that hold chroma.sqlite3, found
# anywhere under /kaggle/input -- unpacked (Kaggle unzips uploads) or still zipped, at any folder depth.
# Folders of .jsonl records (the build's legal_corpus_jsonl) have the same category names but no
# chroma.sqlite3: they are ignored. The 28 Sept run copied that folder instead and ran without a corpus.
import zipfile
INPUT = pathlib.Path("/kaggle/input")
VECTORDB = pathlib.Path("data/legal_corpus_vectordb")
CATEGORIES_WANTED = ("laws", "procedural_rules")
db_roots = sorted({p.parent.parent for p in INPUT.rglob("chroma.sqlite3") if p.parent.name in CATEGORIES_WANTED})
print("database folders found:", [str(r) for r in db_roots] or "none")
shutil.rmtree(VECTORDB, ignore_errors=True)
VECTORDB.mkdir(parents=True)
if db_roots:
    for item in db_roots[0].iterdir():
        (shutil.copytree if item.is_dir() else shutil.copy2)(item, VECTORDB / item.name)
    print("copied", db_roots[0], "->", VECTORDB)
else:
    for z in sorted(INPUT.rglob("*.zip")):
        with zipfile.ZipFile(z) as archive:
            dbs = [n for n in archive.namelist()
                   if n.endswith("chroma.sqlite3") and len(n.split("/")) >= 2 and n.split("/")[-2] in CATEGORIES_WANTED]
            if not dbs:
                continue
            parts = dbs[0].split("/")  # [..., <category>, "chroma.sqlite3"]
            prefix = "/".join(parts[:-2]) + "/" if len(parts) > 2 else ""
            for member in archive.infolist():
                name = member.filename
                if name.startswith(prefix) and not name.endswith("/") and ".." not in name:
                    dest = VECTORDB / name[len(prefix):]
                    dest.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(member) as fsrc, open(dest, "wb") as fdst:
                        shutil.copyfileobj(fsrc, fdst)
            print("unpacked", z, "->", VECTORDB)
            break

CORPUS_CATEGORIES = sorted(c for c in CATEGORIES_WANTED if (VECTORDB / c / "chroma.sqlite3").exists())
assert CORPUS_CATEGORIES, ("NO CORPUS DATABASE FOUND under /kaggle/input -- stopping instead of answering without it. "
                           "Attach the dataset with legal_corpus_vectordb (its laws/ and procedural_rules/ folders "
                           "must contain chroma.sqlite3).")
missing = [c for c in CATEGORIES_WANTED if c not in CORPUS_CATEGORIES]
if missing:
    print(f"[warn] the corpus has no {missing}: questions on those will retrieve nothing from them")

# Every collection must open and hold the chunks the build recorded.
BUILD_INFO = json.loads((VECTORDB / "_build_info.json").read_text()) if (VECTORDB / "_build_info.json").exists() else {}
import chromadb
bad = []
for c in CORPUS_CATEGORIES:
    n = chromadb.PersistentClient(path=str(VECTORDB / c)).get_collection(f"israeli_law_corpus_{c}").count()
    expected = BUILD_INFO.get("categories", {}).get(c, {}).get("chunks")
    print(f"{c:18s} {n:>9,} chunks" + (f" (build: {expected:,})" if expected else ""))
    if n == 0 or (expected and n != expected):
        bad.append(c)
if BUILD_INFO.get("chromadb_version") and BUILD_INFO["chromadb_version"] != chromadb.__version__:
    print(f"[note] built with chromadb {BUILD_INFO['chromadb_version']}, this machine has {chromadb.__version__}")
assert not bad, f"empty or incomplete collections {bad}: re-upload or rebuild the corpus before a long run"
print("corpus OK:", CORPUS_CATEGORIES)

# BM25 (one of the fixes under test): export the lexical copy if the corpus lacks it, then build and
# cache the index now -- and stop if it's missing, so the run doesn't quietly test without it.
no_lexical = [c for c in CORPUS_CATEGORIES if not (VECTORDB / c / f"lexical_{c}.jsonl").exists()]
if no_lexical:
    sh(f'python scripts/legal_data/vectorize.py --vectordb {VECTORDB} --categories {",".join(no_lexical)} --export-lexical',
       check=False)
started = time.time()
sh(f"""python - <<'EOF'
from docslides.legal.corpus_lexical import lexical_index
for category in {CORPUS_CATEGORIES!r}:
    index = lexical_index("{VECTORDB}", category)
    assert index is not None, f"no BM25 index for {{category}}: lexical_{{category}}.jsonl is missing"
    print(category, "BM25:", f"{{len(index.ids):,}} chunks, {{len(index.vocabulary):,}} terms")
EOF""")
print(f"BM25 ready in {time.time() - started:.0f}s")


## 4. Answer: ten questions x five arms, no judge, under the time guard

In [ ]:
import yaml

# The model's config: Ollama, thinking off, the 16k context, retrieval on the GPU, the reasoning trace.
cfg = yaml.safe_load(open("config/config.yaml", encoding="utf-8"))
cfg["legal"]["orchestrator"].update(backend="ollama", base_url="http://localhost:11434", model=MODEL["tag"],
                                    max_model_len=CONTEXT_LENGTH, request_timeout_s=REQUEST_TIMEOUT_S,
                                    supports_thinking=MODEL["thinking"])
cfg["legal"].setdefault("retrieval", {})["device"] = RETRIEVAL_DEVICE
cfg.setdefault("logging", {})["llm_trace_dir"] = str(RUN_DIR / "llm_trace")
CFG = RUN_DIR / "config.yaml"
CFG.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False), encoding="utf-8")
env = {**os.environ, "DOCSLIDES_CONFIG": str(CFG), "DOCSLIDES_LLM_TRACE_DIR": str(RUN_DIR / "llm_trace")}

# Load the model first, at the context size the scripts ask for, so it gets the GPU memory it needs; the
# embedder and reranker then take whichever GPU has room left.
body = json.dumps({"model": MODEL["tag"], "prompt": "", "keep_alive": -1, "options": {"num_ctx": CONTEXT_LENGTH}}).encode()
started = time.time()
urllib.request.urlopen(urllib.request.Request("http://localhost:11434/api/generate", data=body,
                                              headers={"Content-Type": "application/json"}), timeout=1800).read()
print(f"{MODEL['tag']} loaded in {time.time() - started:.0f}s")
sh("ollama ps; nvidia-smi --query-gpu=index,memory.used,memory.total --format=csv")

left_s = int(STOP_ANSWERING_AFTER_H * 3600 - (time.time() - NOTEBOOK_STARTED))
print(f"[time guard] setup took {(time.time() - NOTEBOOK_STARTED) / 60:.0f} min; answering for at most {left_s / 60:.0f} min")
think = " --thinking" if MODEL["thinking"] else ""
sh(f'python scripts/legal_data/ab_fixes.py --model {MODEL["tag"]} --arms {ARMS} --out {RUN_DIR / "ab"} '
   f'--context-length {CONTEXT_LENGTH} --max-tokens {ANSWER_MAX_TOKENS} --timeout-s {REQUEST_TIMEOUT_S} '
   f'--stop-after-s {left_s}{think}', env=env, check=False)

log = RUN_DIR / "ab" / "run.log"
text = log.read_text(encoding="utf-8", errors="replace") if log.exists() else ""
if "legal_reranker_unavailable" in text:
    print("[WARNING] the reranker was unavailable: retrieval used embedding distance only -- not comparable")
if "legal_retrieval_moved_to_cpu" in text:
    print("[note] retrieval ran on the CPU (no room on the GPUs beside the model): same results, slower")


## 5. Comparison and the results zip (for Output)

In [ ]:
# Per-question timing and the judge-free comparison, then the results zip for Output.
import re
for line in text.splitlines():
    if re.match(r"\[\d\d:\d\d:\d\d\] (IL-\d+ \(|variant )", line):
        print(line)
comparison = RUN_DIR / "ab" / "comparison.md"
print(comparison.read_text(encoding="utf-8") if comparison.exists() else "no comparison.md -- see the run log")

traces = " ".join(f'"{p}"' for p in sorted((RUN_DIR / "llm_trace").glob("*.jsonl")))
if traces:
    sh(f'python scripts/llm_trace_report.py {traces} > "{RUN_DIR / "reasoning_report.md"}"', check=False)
sh(f"git log -1 --format='%H %s' > '{RUN_DIR / 'git_commit.txt'}'", check=False)
DEST = f"/kaggle/working/{RUN_NAME}_results.zip"
sh(f"cd /kaggle/working && rm -f {DEST} && zip -qr {DEST} AI-IZ/{RUN_DIR}")
print("results zip:", DEST, f"({pathlib.Path(DEST).stat().st_size / 1e6:.1f} MB) -- send it back for grading")
